**© Copyright AIDENTIFY. All rights reserved.**

# Part 5 | Session 36: MCP(Model Context Protocol) 기반 에이전트 구현

## 🎯 학습 목표

Session 35에서 배운 **Tool Calling**은 LLM이 도구를 호출하는 기본 메커니즘입니다. 하지만 도구가 OpenAI 형식에 종속되고, 다른 LLM 호스트(Claude, Gemini, 로컬 sLLM)에서 재사용하려면 매번 다시 작성해야 한다는 한계가 있습니다.

**MCP(Model Context Protocol)**는 Anthropic이 2024년 11월에 공개한 **LLM ↔ 도구/데이터 소스 통합을 위한 오픈 표준 프로토콜**입니다. 이번 세션에서는 MCP의 구조를 이해하고, **MCP 서버를 직접 구현**하고 **클라이언트로 호출**하는 전체 흐름을 실습합니다.

### 이번 세션에서 배울 내용
- ✅ MCP의 동기와 아키텍처 (Host / Client / Server)
- ✅ MCP의 3대 Primitive: **Tools / Resources / Prompts**
- ✅ 공식 Python SDK로 MCP **서버 직접 구현** (FastMCP)
- ✅ MCP **클라이언트**로 서버에 연결하여 도구 호출
- ✅ LLM(OpenAI)과 MCP 도구 **브리지** 구현
- ✅ 기존 MCP 서버(filesystem 등) 활용 방법

### 실습 환경
- 🔧 GPU 불필요 (API + 로컬 프로세스)
- 🔧 OpenAI API 키 (LLM 브리지 실습용)
- 🔧 Python 3.10+ (MCP SDK 요구사항)

---
## 1️⃣ MCP가 해결하는 문제

### Tool Calling만으로 충분하지 않은 이유

| 항목 | Tool Calling (Session 35) | MCP |
|------|---------------------------|-----|
| 도구 정의 형식 | LLM 제공사별 상이 (OpenAI / Anthropic / Gemini) | **표준 JSON-RPC 2.0** |
| 도구 배포 단위 | 애플리케이션 코드에 내장 | **독립 프로세스 (서버)** |
| 재사용 | 호스트 코드 수정 필요 | 어떤 MCP 호환 호스트든 즉시 사용 |
| 도구 외 기능 | 도구만 | **Resources(파일/DB) + Prompts(템플릿)** |
| 전송 계층 | HTTPS만 | stdio / HTTP+SSE / Streamable HTTP |
| 발견(discovery) | 수동 | `list_tools/resources/prompts`로 자동 |

> 💡 **핵심 비유**: Tool Calling이 "개별 API 통합"이라면, MCP는 "**LLM 생태계의 USB-C**"입니다. 한 번 서버를 만들면 Claude Desktop, Cursor, Continue, Cline 등 어디서나 꽂아 쓸 수 있습니다.

## 2️⃣ MCP 아키텍처

```
┌──────────────────────────┐        ┌───────────────────────┐
│      Host (LLM App)      │        │     MCP Server        │
│  ┌────────────────────┐  │  JSON  │  ┌─────────────────┐  │
│  │     LLM Client     │  │  RPC   │  │     Tools       │  │
│  │  (Claude/OpenAI..) │◀─┼────────┼─▶│  Resources      │  │
│  │                    │  │ stdio  │  │  Prompts        │  │
│  │   MCP Client       │  │  HTTP  │  └─────────────────┘  │
│  └────────────────────┘  │        │                       │
└──────────────────────────┘        └───────────────────────┘
```

### 핵심 컴포넌트

| 구성 요소 | 역할 |
|----------|------|
| **Host** | 사용자 대면 LLM 앱 (Claude Desktop, Cursor, 우리가 만들 노트북 등) |
| **Client** | Host 내부에서 1개 MCP Server와 1:1 연결을 관리하는 모듈 |
| **Server** | 실제 기능을 제공하는 독립 프로세스. Tools/Resources/Prompts를 노출 |
| **Transport** | 통신 채널. `stdio`(로컬 프로세스), `HTTP+SSE`(원격) 등 |

### 3대 Primitive

| Primitive | 설명 | 예시 |
|-----------|------|------|
| **Tools** | LLM이 호출하여 **부수효과**를 일으키는 함수 | DB INSERT, 이메일 발송, 코드 실행 |
| **Resources** | LLM이 **읽기**용으로 받아갈 데이터 (URI로 식별) | 파일 내용, 테이블 스키마, 로그 |
| **Prompts** | 사용자가 명시적으로 트리거하는 **재사용 가능한 프롬프트 템플릿** | `/summarize`, `/code-review` |

---
## 3️⃣ 환경 설정

공식 Python SDK(`mcp`)를 설치합니다. FastMCP는 데코레이터 기반의 고수준 API로 가장 빠르게 서버를 만들 수 있습니다.

In [ ]:
!pip install -q "mcp[cli]>=1.2.0" openai python-dotenv

In [ ]:
import os, sys, json, asyncio, subprocess
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")

if not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key를 입력하세요: ")

# 노트북에서 사용할 작업 디렉토리
WORK_DIR = Path("./output/mcp_demo").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Python: {sys.version.split()[0]}")
print(f"✅ 작업 디렉토리: {WORK_DIR}")

---
## 4️⃣ MCP 서버 직접 구현 (FastMCP)

Session 35의 `get_weather`, `calculate`, `search_web` 함수를 MCP 표준으로 재패키징합니다.

MCP 서버는 **별도 프로세스**로 실행되므로, 서버 코드를 `.py` 파일로 저장합니다. `stdio` 전송을 사용하면 부모 프로세스(노트북)가 자식 프로세스(서버)를 띄우고 표준입출력으로 통신합니다.

> 💡 **FastMCP**는 데코레이터(`@mcp.tool()`)로 도구를 등록하면 함수 시그니처와 docstring을 자동으로 JSON Schema로 변환해 줍니다. Session 35에서 손으로 작성한 스키마를 자동화해 주는 셈입니다.

In [ ]:
# =====================================================
# 📝 MCP 서버 코드를 파일로 저장
# =====================================================
server_code = '''
"""
demo_mcp_server.py - 실습용 MCP 서버
Tools, Resources, Prompts를 모두 노출하는 미니멀 예제.
실행: python demo_mcp_server.py  (stdio 모드)
"""
import random
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-server")

# ----- Tools -----
@mcp.tool()
def get_weather(city: str) -> dict:
    """주어진 도시의 현재 날씨를 반환합니다."""
    conditions = ["맑음", "흐림", "비", "눈", "구름 많음"]
    return {
        "city": city,
        "temperature": random.randint(-5, 35),
        "condition": random.choice(conditions),
        "humidity": random.randint(30, 90),
    }

@mcp.tool()
def calculate(expression: str) -> dict:
    """수학 표현식을 계산합니다. 사칙연산, 거듭제곱 등을 지원합니다."""
    try:
        # 보안: 내장 함수 차단
        result = eval(expression, {"__builtins__": {}}, {})
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "error": str(e)}

@mcp.tool()
def search_notes(query: str, limit: int = 3) -> list:
    """로컬 메모 저장소에서 키워드와 일치하는 메모를 검색합니다."""
    notes = [
        {"id": 1, "title": "MCP 개념", "body": "MCP는 LLM-도구 통합의 표준 프로토콜이다."},
        {"id": 2, "title": "A2A 개념", "body": "A2A는 에이전트 간 통신을 위한 Google의 프로토콜이다."},
        {"id": 3, "title": "LangGraph", "body": "LangGraph는 그래프 기반 에이전트 오케스트레이션 도구이다."},
        {"id": 4, "title": "LoRA", "body": "LoRA는 저랭크 어댑터를 활용한 효율적 파인튜닝 기법이다."},
    ]
    matches = [n for n in notes if query.lower() in n["title"].lower() or query.lower() in n["body"].lower()]
    return matches[:limit]

# ----- Resources -----
@mcp.resource("config://app")
def get_app_config() -> str:
    """애플리케이션의 현재 설정을 JSON으로 반환합니다."""
    import json
    return json.dumps({
        "version": "1.0.0",
        "model": "gpt-4o-mini",
        "features": ["weather", "calc", "notes"],
    }, ensure_ascii=False, indent=2)

@mcp.resource("notes://{note_id}")
def get_note(note_id: str) -> str:
    """ID로 특정 메모의 전체 본문을 가져옵니다."""
    notes = {
        "1": "MCP 개념: Model Context Protocol은 Anthropic이 2024년에 발표한 LLM-도구 통합의 오픈 표준입니다.",
        "2": "A2A 개념: Agent-to-Agent 프로토콜은 Google이 2025년에 발표한 멀티에이전트 통신 표준입니다.",
    }
    return notes.get(note_id, f"메모 {note_id}를 찾을 수 없습니다.")

# ----- Prompts -----
@mcp.prompt()
def review_note(note_id: str) -> str:
    """특정 메모를 검토하기 위한 프롬프트를 생성합니다."""
    return (
        f"`notes://{note_id}` 리소스를 읽고, 핵심을 3줄 이내로 요약한 뒤 "
        f"개선할 점 1가지를 제안해 주세요."
    )

if __name__ == "__main__":
    mcp.run()  # 기본값: stdio transport
'''

server_path = WORK_DIR / "demo_mcp_server.py"
server_path.write_text(server_code, encoding="utf-8")
print(f"✅ 서버 코드 저장: {server_path}")
print(f"📌 크기: {server_path.stat().st_size} bytes")

### 서버 코드 해설

- **`FastMCP("demo-server")`**: 서버 이름. Host가 여러 서버를 구분할 때 사용
- **`@mcp.tool()`**: 함수 시그니처(`city: str`)와 docstring이 자동으로 JSON Schema로 변환됨
- **`@mcp.resource("uri://pattern")`**: URI 템플릿 지원. `{note_id}`는 동적 파라미터
- **`@mcp.prompt()`**: 사용자가 슬래시 커맨드처럼 호출하는 템플릿
- **`mcp.run()`**: 기본적으로 `stdio` 전송 모드로 서버 실행 → 부모 프로세스가 표준입출력으로 통신

---
## 5️⃣ MCP 클라이언트 구현 — 서버에 연결

이제 노트북에서 클라이언트를 만들어 방금 작성한 서버에 연결합니다.

### MCP 핸드셰이크 흐름
```
1. stdio_client(server_params)  → 자식 프로세스 spawn (python demo_mcp_server.py)
2. ClientSession(read, write)    → JSON-RPC 채널 생성
3. session.initialize()          → 프로토콜 버전 협상
4. session.list_tools()          → 노출된 도구 목록 조회
5. session.call_tool(name, args) → 도구 호출 (LLM이 결정한 결과)
```

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,           # 현재 파이썬 인터프리터로 서버 실행
    args=[str(server_path)],
)

async def list_server_capabilities():
    """서버가 노출하는 Tools/Resources/Prompts를 모두 나열"""
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            tools = await session.list_tools()
            resources = await session.list_resources()
            res_templates = await session.list_resource_templates()
            prompts = await session.list_prompts()
            
            return tools, resources, res_templates, prompts

tools, resources, res_templates, prompts = await list_server_capabilities()

print("🔧 Tools:")
for t in tools.tools:
    print(f"  - {t.name}: {t.description}")

print("\n📄 Resources (구체 URI):")
for r in resources.resources:
    print(f"  - {r.uri}: {r.description or r.name}")

print("\n📄 Resource Templates (URI 패턴):")
for r in res_templates.resourceTemplates:
    print(f"  - {r.uriTemplate}: {r.description or r.name}")

print("\n💬 Prompts:")
for p in prompts.prompts:
    print(f"  - {p.name}: {p.description}")

### 도구 호출

`call_tool(name, arguments)`로 서버의 함수를 원격 호출합니다. 결과는 `content` 배열에 들어 있습니다.

In [ ]:
async def call_demo_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 1. 날씨 도구 호출
            weather = await session.call_tool("get_weather", {"city": "서울"})
            print("🌤️  get_weather(서울):")
            for c in weather.content:
                print(f"   {c.text}")
            
            # 2. 계산기 호출
            calc = await session.call_tool("calculate", {"expression": "(2 + 3) * 7"})
            print("\n🔢 calculate('(2+3)*7'):")
            for c in calc.content:
                print(f"   {c.text}")
            
            # 3. 노트 검색
            notes = await session.call_tool("search_notes", {"query": "MCP", "limit": 2})
            print("\n📝 search_notes('MCP'):")
            for c in notes.content:
                print(f"   {c.text}")
            
            # 4. Resource 읽기
            cfg = await session.read_resource("config://app")
            print("\n📄 read_resource('config://app'):")
            for c in cfg.contents:
                print(f"   {c.text}")
            
            # 5. URI 템플릿 리소스 읽기
            note = await session.read_resource("notes://1")
            print("\n📄 read_resource('notes://1'):")
            for c in note.contents:
                print(f"   {c.text}")
            
            # 6. Prompt 가져오기
            prompt = await session.get_prompt("review_note", {"note_id": "1"})
            print("\n💬 get_prompt('review_note', note_id=1):")
            for msg in prompt.messages:
                print(f"   [{msg.role}] {msg.content.text}")

await call_demo_tools()

---
## 6️⃣ LLM ↔ MCP 브리지 — 진짜 에이전트 만들기

지금까지는 우리가 "수동으로" 도구를 호출했습니다. 이제 **LLM이 사용자 질문을 보고 어떤 MCP 도구를 호출할지 결정**하도록 만들어 봅시다.

흐름:
```
사용자 질문 → MCP list_tools() → OpenAI 형식 변환 → LLM이 도구 선택
          → MCP call_tool()로 실행 → 결과를 LLM에 전달 → 최종 응답
```

여기서 MCP의 장점이 드러납니다: 서버 코드는 그대로 두고 **LLM만 갈아끼울 수 있습니다**(OpenAI → Claude → 로컬 sLLM).

In [ ]:
from openai import OpenAI

openai_client = OpenAI()
MODEL = "gpt-4o-mini"

def mcp_tool_to_openai(mcp_tool) -> dict:
    """MCP 도구 정의를 OpenAI Function Calling 형식으로 변환"""
    return {
        "type": "function",
        "function": {
            "name": mcp_tool.name,
            "description": mcp_tool.description,
            "parameters": mcp_tool.inputSchema,  # JSON Schema 호환
        },
    }

# 변환 예시 확인
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tool_list = await session.list_tools()
        openai_tools = [mcp_tool_to_openai(t) for t in tool_list.tools]

print("🔄 MCP → OpenAI 변환 결과:\n")
print(json.dumps(openai_tools[0], indent=2, ensure_ascii=False))

In [ ]:
async def mcp_agent(user_question: str, model: str = MODEL):
    """
    MCP 서버 + OpenAI LLM을 연결한 에이전트.
    LLM이 도구 호출이 필요 없다고 판단할 때까지 반복(ReAct 루프).
    """
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 1. MCP 도구 목록 → OpenAI 형식
            tool_list = await session.list_tools()
            openai_tools = [mcp_tool_to_openai(t) for t in tool_list.tools]
            
            messages = [
                {"role": "system", "content": "당신은 MCP 도구를 활용하는 어시스턴트입니다. 한국어로 답변하세요."},
                {"role": "user", "content": user_question},
            ]
            
            print(f"👤 사용자: {user_question}\n")
            
            for step in range(5):  # 최대 5 라운드
                resp = openai_client.chat.completions.create(
                    model=model, messages=messages, tools=openai_tools, tool_choice="auto",
                )
                msg = resp.choices[0].message
                messages.append(msg)
                
                if not msg.tool_calls:
                    print(f"🤖 최종 응답: {msg.content}")
                    return msg.content
                
                # 도구 호출 실행 (MCP 서버에 위임)
                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments)
                    print(f"  📞 MCP 호출: {tc.function.name}({args})")
                    result = await session.call_tool(tc.function.name, args)
                    # MCP content를 문자열로 평탄화
                    payload = "\n".join(c.text for c in result.content if hasattr(c, "text"))
                    print(f"     ↳ {payload[:120]}")
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": payload,
                    })
            
            print("⚠️  최대 라운드 도달")
            return None

# 테스트 1: 단일 도구
print("=" * 60)
await mcp_agent("서울 날씨 알려줘.")
print()

# 테스트 2: 멀티 도구
print("=" * 60)
await mcp_agent("부산 날씨를 알아본 다음, 12 * 345는 얼마인지 계산해줘.")
print()

# 테스트 3: 도구 불필요
print("=" * 60)
await mcp_agent("안녕! 너 누구야?")

---
## 7️⃣ 기존 MCP 서버 활용 — 생태계의 힘

MCP의 진짜 가치는 **이미 만들어진 서버 생태계**에 있습니다. 우리가 코드를 한 줄도 안 짜도, npm/pip로 설치만 하면 됩니다.

### 대표적인 공식/커뮤니티 서버

| 서버 | 기능 | 설치 |
|------|------|------|
| `@modelcontextprotocol/server-filesystem` | 디렉토리 읽기/쓰기 | `npx -y @modelcontextprotocol/server-filesystem <path>` |
| `@modelcontextprotocol/server-github` | GitHub 이슈/PR 조작 | `npx -y @modelcontextprotocol/server-github` |
| `@modelcontextprotocol/server-postgres` | PostgreSQL 읽기 전용 쿼리 | `npx -y @modelcontextprotocol/server-postgres <conn>` |
| `mcp-server-sqlite` | SQLite 분석 | `uvx mcp-server-sqlite --db-path <file>` |
| `mcp-server-fetch` | URL fetch + HTML→Markdown | `uvx mcp-server-fetch` |

> 💡 **검색**: https://github.com/modelcontextprotocol/servers 와 https://mcp.so 에서 더 많은 서버를 찾을 수 있습니다.

### 예시: filesystem 서버 연결 (Node가 설치되어 있을 때)

아래 코드는 참고용입니다. 실행하려면 `node`와 `npx`가 설치되어 있어야 합니다.

In [ ]:
# =====================================================
# 📂 filesystem MCP 서버에 연결하는 코드 예시 (옵션)
# Node + npx 가 있을 때만 동작합니다.
# =====================================================
import shutil

if shutil.which("npx") is None:
    print("⚠️  npx가 없어 이 셀은 스킵합니다. (Node.js 설치 후 재시도)")
else:
    # WORK_DIR을 노출하는 filesystem 서버 띄우기
    fs_params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", str(WORK_DIR)],
    )
    
    async with stdio_client(fs_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("🔧 filesystem 서버가 노출하는 도구:")
            for t in tools.tools:
                print(f"  - {t.name}")
            
            # 예시: 디렉토리 목록
            result = await session.call_tool("list_directory", {"path": str(WORK_DIR)})
            print("\n📂 list_directory 결과:")
            for c in result.content:
                print(f"   {c.text}")

### Claude Desktop / Cursor에 우리 서버 등록하기

방금 만든 `demo_mcp_server.py`는 Claude Desktop에서도 그대로 쓸 수 있습니다. 설정 파일에 다음을 추가하면 됩니다:

**macOS**: `~/Library/Application Support/Claude/claude_desktop_config.json`  
**Windows**: `%APPDATA%\Claude\claude_desktop_config.json`

```json
{
  "mcpServers": {
    "demo": {
      "command": "python",
      "args": ["/절대/경로/demo_mcp_server.py"]
    }
  }
}
```

재시작 후 Claude에게 "서울 날씨 알려줘"라고 하면 우리 서버의 `get_weather`를 호출합니다 — **노트북 코드 한 줄 안 바꾸고 다른 호스트에서 재사용**되는 것이 MCP의 핵심 가치입니다.

---
## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 |
|------|------|
| **MCP 동기** | LLM ↔ 도구 통합의 "USB-C" — 한 번 짜면 어디서나 재사용 |
| **아키텍처** | Host(앱) ↔ Client(연결자) ↔ Server(기능 제공자) |
| **3대 Primitive** | Tools(부수효과) / Resources(읽기 데이터) / Prompts(템플릿) |
| **FastMCP** | 데코레이터 기반 서버 작성 — 시그니처/docstring 자동 스키마화 |
| **stdio 전송** | 부모-자식 프로세스 간 JSON-RPC 통신 (가장 단순) |
| **LLM 브리지** | `mcp_tool → OpenAI tool` 변환 한 번으로 모든 함수 사용 가능 |
| **생태계** | filesystem, github, postgres 등 즉시 사용 가능한 공식 서버 다수 |

### Tool Calling (Session 35) vs MCP (Session 36)

- Tool Calling은 **"어떻게 LLM이 함수를 부를까?"** 의 답
- MCP는 **"누가 어떤 함수를 제공할지 어떻게 표준화할까?"** 의 답
- 둘은 경쟁 관계가 아니라 **레이어가 다른 보완 관계** — MCP 클라이언트가 결국 Tool Calling으로 LLM과 대화

### 다음 노트북
👉 **Session 37** (`37_a2a_protocol.ipynb`): A2A(Agent-to-Agent) 프로토콜 — 도구가 아닌 **에이전트끼리** 표준화된 방식으로 통신하기

In [ ]:
print("Session 36: MCP 기반 에이전트 구현 실습 완료!")
print("\n핵심 산출물:")
print(f"  - MCP 서버: {server_path}")
print(f"  - 클라이언트: 이 노트북의 mcp_agent() 함수")
print(f"\n다음 단계 추천:")
print("  1. demo_mcp_server.py에 도구/리소스 추가해 보기")
print("  2. Claude Desktop에 서버 등록해서 동일하게 동작하는지 확인")
print("  3. 공식 서버 디렉토리에서 본인 업무에 필요한 서버 찾아보기")
print("     → https://github.com/modelcontextprotocol/servers")